# 2장. VS Code에서 시작하는 데이터 분석 환경

환경 점검 실습 노트북입니다. 책 Chapter 02의 Part 13~14를 따라
터미널 Python · VS Code 인터프리터 · Notebook 커널이 같은 `.venv`를 가리키는지,
그리고 `data/raw`의 CSV를 실제로 읽을 수 있는지 확인합니다.

## 실행 방법

1. 오른쪽 위 **Select Kernel → Python Environments → 프로젝트 `.venv`** 선택
2. **Restart Kernel → Run All** 로 처음부터 전체 실행

## 성공 기준

* Python 실행 경로에 `.venv` 가 포함된다.
* `customer_path.exists()` 가 `True` 다.
* `customers.head()` 가 표로 출력되고 `customers.shape` 가 `(150, 6)` 이다.

## 55. 커널 선택과 프로젝트 루트

상대 경로는 노트북 파일 위치가 아니라 **현재 작업 폴더**를 기준으로 해석됩니다.
이 노트북은 `notebooks/ch02/` 에 있으므로, 실행 위치가 어디든
상위 폴더로 올라가며 `data` 폴더(또는 `.git`)가 있는 지점을 찾아
프로젝트 루트로 사용합니다.

In [ ]:
from pathlib import Path


def find_project_root(start=None):
    """상위로 올라가며 data/raw 폴더가 실제로 있는 곳을 프로젝트 루트로 본다.

    1순위: data/raw 폴더가 있는 상위 폴더
    2순위: data 폴더나 .git 이 있는 상위 폴더
    둘 다 없으면 시작 위치를 그대로 돌려준다.
    """
    path = Path(start or Path.cwd()).resolve()
    chain = (path, *path.parents)
    for candidate in chain:
        if (candidate / "data" / "raw").is_dir():
            return candidate
    for candidate in chain:
        if (candidate / "data").is_dir() or (candidate / ".git").exists():
            return candidate
    return path


project_root = find_project_root()
data_dir = project_root / "data" / "raw"
customer_path = data_dir / "customers.csv"

print("프로젝트 루트:", project_root)
print("데이터 폴더:", data_dir)
print("데이터 폴더 존재:", data_dir.is_dir())

## 56. 첫 번째 셀: 환경 확인

현재 커널이 어느 Python 을 사용하는지 확인합니다. 경로에 `.venv` 가 포함되어야 합니다.

In [ ]:
import sys

print("Python 실행 경로:", sys.executable)
print("Python 버전:", sys.version.split()[0])
print("현재 작업 폴더:", Path.cwd())
print(".venv 사용 여부:", ".venv" in sys.executable)

## 57. 두 번째 셀: 패키지 확인

오류 없이 버전이 출력되면 현재 커널에서 해당 패키지를 사용할 수 있습니다.
`ModuleNotFoundError` 가 나면 터미널의 `.venv` 와 커널이 다른 환경일 가능성이 큽니다.

In [ ]:
import pandas as pd
import numpy as np

print("pandas:", pd.__version__)
print("numpy:", np.__version__)

for name in ["matplotlib", "seaborn", "sklearn", "faker", "dotenv"]:
    try:
        __import__(name)
        print(f"{name}: import 성공")
    except ModuleNotFoundError as error:
        print(f"{name}: 설치 필요 -> {error}")

## 58. 세 번째 셀: 파일 존재 확인

파일이 없으면 프로젝트 루트에서 `python scripts/generate_sample_data.py` 를 먼저 실행합니다.

In [ ]:
print("데이터 폴더 존재:", data_dir.exists())
print("고객 파일 경로:", customer_path.resolve())
print("고객 파일 존재:", customer_path.exists())

if data_dir.exists():
    print("폴더 안 파일:", [p.name for p in sorted(data_dir.glob("*.csv"))])

## 59. 네 번째 셀: CSV 읽기

출력이 없어도 정상입니다. 변수 `customers` 에 DataFrame 이 저장됩니다.
생성 파일은 `utf-8-sig` 로 저장되어 있어 한글이 깨지면 `encoding="utf-8-sig"` 를 지정합니다.

In [ ]:
customers = pd.read_csv(customer_path, encoding="utf-8-sig")

## 60. 다섯 번째 셀: 첫 표 확인

실제 컬럼: `customer_id`, `name`, `gender`, `age`, `city`, `signup_date`.
기본 생성 기준 크기는 `(150, 6)` 입니다.

In [ ]:
customers.head()

In [ ]:
print("shape:", customers.shape)
customers.info()

## 61. Chapter 02 최종 성공 화면

다음 세 결과가 모두 확인되어야 합니다.

1. `customer_path.exists()` → `True`
2. `pd.read_csv(customer_path)` → 오류 없음
3. `customers.head()` → 표 출력

이는 Python · 가상환경 · pandas · VS Code · Notebook 커널 · 현재 작업 폴더 · CSV 경로가
모두 연결되었다는 뜻입니다. 데이터 품질과 컬럼 의미 검증은 Chapter 03에서 진행합니다.

## 62. Notebook이 이상할 때

`Restart Kernel → Run All` 로 처음부터 다시 실행합니다.
그래도 문제가 있으면 아래 셀로 현재 커널의 실행 경로와 작업 폴더를 확인합니다.

In [ ]:
print(sys.executable)
print(Path.cwd())

## 63. .gitignore 확인 (터미널에서 실행)

`.venv` 와 `.env` 가 Git 추적에서 제외되는지 프로젝트 루트 터미널에서 확인합니다.

```powershell
git check-ignore -v .venv
git check-ignore -v .env
```

제외되고 있다면 어떤 규칙 때문에 제외되는지 함께 출력됩니다.

## 68. 최종 환경 진단 셀

노트북 마지막에서 전체 환경을 한 번에 점검합니다.

In [ ]:
import sys
from pathlib import Path

import pandas as pd


def find_project_root(start=None):
    path = Path(start or Path.cwd()).resolve()
    chain = (path, *path.parents)
    for candidate in chain:
        if (candidate / "data" / "raw").is_dir():
            return candidate
    for candidate in chain:
        if (candidate / "data").is_dir() or (candidate / ".git").exists():
            return candidate
    return path


project_root = find_project_root()
data_dir = project_root / "data" / "raw"
customer_path = data_dir / "customers.csv"

print("Python:", sys.executable)
print("프로젝트:", project_root)
print("pandas:", pd.__version__)
print("데이터 폴더 존재:", data_dir.is_dir())
print("customers.csv 존재:", customer_path.is_file())

if customer_path.is_file():
    customers = pd.read_csv(customer_path, encoding="utf-8-sig")
    print("customers shape:", customers.shape)
    display(customers.head())
else:
    print("[안내] customers.csv 를 찾지 못했습니다. scripts/generate_sample_data.py 를 먼저 실행하세요.")

## 69. 루트 후보 비교 (경로 진단)

현재 작업 폴더부터 상위로 올라가며 후보 경로를 나열하고,
각 후보가 **프로젝트 루트인지**(= `data` 폴더 또는 `.git` 보유)와
**데이터 폴더 존재 여부**를 확인합니다.
마지막에 실제로 선택된 프로젝트 루트와 데이터 폴더 경로를 출력합니다.

In [ ]:
from pathlib import Path

cwd = Path.cwd().resolve()
candidates = [cwd, *cwd.parents][:4]

print("현재 작업 폴더:", cwd, "\n")

# 모든 후보 경로는 그대로 출력한다.
valid_roots = []
for i, path in enumerate(candidates, start=1):
    is_root = (path / "data").is_dir() or (path / ".git").exists()
    has_data = (path / "data" / "raw").is_dir()
    print(f"[루트 후보 {i}]")
    print("  경로:", path)
    print("  프로젝트 루트 여부:", is_root)
    print("  데이터 폴더 존재 여부:", has_data)
    print()
    if has_data:
        valid_roots.append(path)

# 실제 선택은 데이터 폴더가 존재하는(True) 후보 중에서만 한다 -> 이후 셀에서 오류가 나지 않는다.
print("데이터 폴더가 존재하는 후보:")
for path in valid_roots:
    print("  -", path, "->", path / "data" / "raw")

if valid_roots:
    selected_root = valid_roots[0]
    selected_data = selected_root / "data" / "raw"
    print("\n=> 선택된 프로젝트 루트:", selected_root)
    print("=> 데이터 폴더 경로:", selected_data)
    print("=> 데이터 폴더 존재 여부:", selected_data.is_dir())
else:
    selected_root = None
    selected_data = None
    print("\n[경고] data/raw 폴더를 가진 상위 폴더를 찾지 못했습니다.")
    print("       scripts/generate_sample_data.py 를 실행했는지 확인하세요.")